← [12 - Arithmetique reelle (Python)](Z3-12-Real-Arithmetic-Python.ipynb) | [README Z3-API](README.md) | [Version Rust](Z3-13-UnsatCores-Rust.ipynb) | [13b - UNSAT cores : le MUS >>](Z3-13b-UnsatCores-MUS-CSharp.ipynb)
# 13. UNSAT cores : expliquer l'insatisfiabilite (le ' pourquoi ')
Jusqu'ici, le solveur repondait a une question binaire : *ce système a-t-il une solution ?* -> **SAT** (oui, voici un temoin) ou **UNSAT** (non, prouve impossible). C'est un **verdict**, pas une **explication**.
Pour un ingenieur qui **ecrit** la specification, UNSAT est souvent un bug de modelisation : une contrainte trop forte, ou deux contraintes incompatibles. La reponse ' impossible ' ne suffit pas - il faut savoir **lesquelles** des contraintes se contredisent. C'est le rôle du **noyau d'insatisfiabilite** (UNSAT core) : le sous-ensemble **minimal** de contraintes responsable du conflit. Z3 passe du diagnostic binaire au diagnostic chirurgical.
> Jumeau C# du notebook Python [`Z3-13-UnsatCores-Python.ipynb`](Z3-13-UnsatCores-Python.ipynb). (Le bloc DSL `Explain()` du C#, spécifique a Z3.Linq, n'est pas porte ; l'API brute `AssertAndTrack` + `UnsatCore` de Microsoft.Z3 fait le même travail.)

**Voir aussi** : le carnet cousin [`17_UnsatCores.ipynb`](../Z3-Linq2Z3/17_UnsatCores.ipynb) présente la même notion avec l'API déclarative Z3.Linq.
## Source

Ce notebook s'inscrit dans la série ancrée par **Yurichev, Dennis — *SAT/SMT by Example* (édition publique du 12 février 2023, 666 pages)**, §3.13.2 Unsat core (p. 90), Ch. 3 Equations — UNSAT cores, AssertAndTrack, diagnostic (folio du corps de l'édition archivée). Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.


In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Linq;
using System.Collections.Generic;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);
Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);


Installing Packages Microsoft.Z3

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


## 1. Du verdict a l'explication

Commencons par un système trivialement contradictoire. Le solveur dit **UNSAT** - mais ne dit pas (encore) POURQUOI. C'est le verdict brut.


In [2]:
// Rappel : un systeme contradictoire est UNSAT (verdict, sans explication)
var ctx0 = new Context();
var a = ctx0.MkBoolConst("a");
var b = ctx0.MkBoolConst("b");
var s0 = ctx0.MkSolver();
s0.Assert(ctx0.MkImplies(a, b));   // a -> b
s0.Assert(a);                      // a
s0.Assert(ctx0.MkNot(b));           // !b  (contredit a -> b)
Console.WriteLine("' a->b, a, !b ' : " + s0.Check() + "  (verdict : impossible)");
Console.WriteLine("-> Z3 dit NON, mais ne dit pas (encore) POURQUOI.");


' a->b, a, !b ' : UNSATISFIABLE  (verdict : impossible)


-> Z3 dit NON, mais ne dit pas (encore) POURQUOI.


**Lecture chiffree — un systeme ou le core serait TOUT.** `' a->b, a, !b ' : unsat` : le verdict brut tombe en une ligne. Mais ce petit systeme a une propriete que les exemples suivants n'auront pas : aucune de ses trois contraintes n'est superflue. Chaque paire prise seule est satisfiable — `{a->b, a}` tient avec `b` vrai, `{a->b, !b}` tient avec `a` faux, `{a, !b}` tient sans l'implication. Autrement dit, un core etiquete de ce systeme contiendrait les TROIS contraintes : le conflit nait de la combinaison complete, pas d'une paire. C'est la frontiere utile a connaitre avant la section suivante — l'etiquetage `AssertAndTrack` ne reduit que ce qui peut l'etre : quand chaque contrainte participe a la refutation, le core egale le systeme, et l'explication reste a ecrire a la main.


## 2. L'API `AssertAndTrack` : etiqueter chaque contrainte
Pour obtenir une explication, on fournit les contraintes comme des **hypothese traceables** : `solver.AssertAndTrack(formule, etiquette)` ou `etiquette` est un `BoolExpr` servant de nom. Après un `unsat`, `solver.UnsatCore` renvoie le sous-ensemble **minimal** d'etiquettes responsable du conflit.
Trois contraintes sur `x` : `x >= 3`, `x <= 5`, `x = 10`. Lesquelles se contredisent ?


In [3]:
// Trois contraintes sur x : lesquelles se contredisent ?
var ctx6 = new Context();
var x = ctx6.MkIntConst("x");
var s1 = ctx6.MkSolver();
var c1 = ctx6.MkBoolConst("x_ge_3");   // etiquette pour x >= 3
var c2 = ctx6.MkBoolConst("x_le_5");   // etiquette pour x <= 5
var c3 = ctx6.MkBoolConst("x_eq_10");  // etiquette pour x = 10
s1.AssertAndTrack(ctx6.MkGe(x, ctx6.MkInt(3)), c1);
s1.AssertAndTrack(ctx6.MkLe(x, ctx6.MkInt(5)), c2);
s1.AssertAndTrack(ctx6.MkEq(x, ctx6.MkInt(10)), c3);
var st1 = s1.Check();
Console.WriteLine("' x>=3, x<=5, x=10 ' : " + st1);
Console.WriteLine();
if (st1 == Status.UNSATISFIABLE)
{
    Console.WriteLine("UNSAT CORE (contraintes conflictuelles, sous-ensemble MINIMAL) :");
    var core1 = s1.UnsatCore.Select(c => c.ToString()).OrderBy(c => c).ToArray();
    foreach (var c in core1)
    {
        Console.WriteLine("  - " + c);
    }
    Console.WriteLine("-> Z3 ecarte 'x>=3' du core : cette contrainte est irrelevant pour le conflit.");
}


' x>=3, x<=5, x=10 ' : UNSATISFIABLE


UNSAT CORE (contraintes conflictuelles, sous-ensemble MINIMAL) :


  - x_eq_10


  - x_le_5


-> Z3 ecarte 'x>=3' du core : cette contrainte est irrelevant pour le conflit.


**Lecture chiffree — la minimalite verifiee dans les DEUX sens.** Core rendu : `x_le_5`, `x_eq_10` — la section 3 montrera que la paire suffit. L'autre sens est tout aussi testable et complete la definition : chaque membre du core pris individuellement est NECESSAIRE. Retirer `x_le_5` : `{x>=3, x=10}` est satisfiable (`x = 10` verifie les deux). Retirer `x_eq_10` : `{x>=3, x<=5}` est satisfiable (tout `x` de 3 a 5). Aucune des deux ne peut partir — le core est irreductible dans les deux directions : suffisant (paire UNSAT) et necessaire (chaque retrait redevient SAT). C'est la definition complete du MUS que la section 7 formalisera, deja realisee ici sur un exemple ou l'ecart se voit a l'oeil nu : `x = 10` est a 5 unites de la borne `x <= 5`.


## 3. Minimalite du core : verifier que la paire suffit

Le core est **minimal** : retirer une seule contrainte du core le rend satisfiable. Verifions que la paire `{x<=5, x=10}` (sans `x>=3`) est **déjà** UNSAT - ce qui confirme que Z3 avait raison d'exclure `x>=3`.


In [4]:
// Verifions : {x<=5, x=10} SEUL (sans x>=3) est-il deja UNSAT ?
var ctx9 = new Context();
var y = ctx9.MkIntConst("y");
var s2 = ctx9.MkSolver();
var p1 = ctx9.MkBoolConst("y_le_5");
var p2 = ctx9.MkBoolConst("y_eq_10");
s2.AssertAndTrack(ctx9.MkLe(y, ctx9.MkInt(5)), p1);
s2.AssertAndTrack(ctx9.MkEq(y, ctx9.MkInt(10)), p2);
var st2 = s2.Check();
Console.WriteLine("' y<=5, y=10 ' (sans y>=3) : " + st2);
if (st2 == Status.UNSATISFIABLE)
{
    Console.WriteLine("-> CONFIRME : la paire {x<=5, x=10} suffit au conflit.");
    Console.WriteLine("   Z3 avait donc raison d'EXCLURE 'x>=3' du core : elle est irrelevant.");
}


' y<=5, y=10 ' (sans y>=3) : UNSATISFIABLE


-> CONFIRME : la paire {x<=5, x=10} suffit au conflit.


   Z3 avait donc raison d'EXCLURE 'x>=3' du core : elle est irrelevant.


## 4. Application : specification surcontrainte - laquelle relacher ?

Cas realiste : un planning ou 5 contraintes s'accumulent. L'une d'elles (`h = 12`, demande de l'animateur) contredit une autre (`h != 12`, pause dejeuner). Le core identifie **exactement** les 2 coupables parmi les 5 - les autres (bornes, redondances) sont compatibles et ignorees. L'action correctrice en decoule directement : relacher l'une des deux contraintes coupables.


In [5]:
// Specification surcontrainte : laquelle relacher ?
var ctx11 = new Context();
var h = ctx11.MkIntConst("h");
var s3 = ctx11.MkSolver();
var n1 = ctx11.MkBoolConst("h_ge_9");     // 1. h >= 9
var n2 = ctx11.MkBoolConst("h_le_17");    // 2. h <= 17
var n3 = ctx11.MkBoolConst("h_ne_12");    // 3. h != 12 (pause dejeuner)
var n4 = ctx11.MkBoolConst("h_eq_12");    // 4. h = 12 (animateur)  <-- conflit
var n5 = ctx11.MkBoolConst("h_ge_9b");    // 5. h >= 9 (redondant)
s3.AssertAndTrack(ctx11.MkGe(h, ctx11.MkInt(9)), n1);
s3.AssertAndTrack(ctx11.MkLe(h, ctx11.MkInt(17)), n2);
s3.AssertAndTrack(ctx11.MkNot(ctx11.MkEq(h, ctx11.MkInt(12))), n3);
s3.AssertAndTrack(ctx11.MkEq(h, ctx11.MkInt(12)), n4);
s3.AssertAndTrack(ctx11.MkGe(h, ctx11.MkInt(9)), n5);
var st3 = s3.Check();
Console.WriteLine("Specification (5 contraintes) : " + st3);
Console.WriteLine();
if (st3 == Status.UNSATISFIABLE)
{
    Console.WriteLine("Core = les contraintes qui PARTICIPENT au conflit :");
    var core3 = s3.UnsatCore.Select(c => c.ToString()).OrderBy(c => c).ToArray();
    foreach (var c in core3)
    {
        Console.WriteLine("  COUPABLE : " + c);
    }
    Console.WriteLine();
    Console.WriteLine("-> Sur 5 contraintes, " + core3.Length + " seulement sont en conflit.");
    Console.WriteLine("   Les autres (h>=9, h<=17, h>=9 redondant) sont COMPATIBLES -> Z3 les ignore.");
    Console.WriteLine("   Action : relacher la contrainte 'h=12' (animateur) OU 'h!=12' (pause).");
}


Specification (5 contraintes) : UNSATISFIABLE


Core = les contraintes qui PARTICIPENT au conflit :


  COUPABLE : h_eq_12


  COUPABLE : h_ne_12


-> Sur 5 contraintes, 2 seulement sont en conflit.


   Les autres (h>=9, h<=17, h>=9 redondant) sont COMPATIBLES -> Z3 les ignore.


   Action : relacher la contrainte 'h=12' (animateur) OU 'h!=12' (pause).


**Lecture chiffree — les deux relachements ne coutent pas le meme prix.** `Sur 5 contraintes, 2 seulement sont en conflit` — mais les deux manieres de guerir ne sont pas equivalentes, et le systeme restant les chiffre. Les trois contraintes innocentes bornent `h` dans l'intervalle [9, 17], soit 17 - 9 + 1 = 9 valeurs entieres. Relacher `h=12` (garder la pause dejeuner) : 9 - 1 = 8 solutions restent. Relacher `h!=12` (garder la demande de l'animateur) : exactement 1 solution, `h = 12`. Le core dit QUELLES contraintes sont en conflit ; compter ce que chaque relachement laisse voir est le pas supplementaire qui transforme le diagnostic en decision — ici, l'option pause dejeuner preserve 8 des 9 plannings possibles, l'option animateur en fige un seul.


## 5. Innocence prouvee : une contrainte non impliquee ecartee du core
Un dernier temoin de la minimalite : sur un système a 3 contraintes (`x == 1`, `x >= 0` innocente, `x == 2`), le core minimal est `{x==1, x==2}`. La contrainte `x >= 0` est **compatible** avec les deux (un x valant 1 ou 2 verifie x >= 0) - elle n'est pas impliquee dans le conflit, donc Z3 l'ecarte du core. C'est ce qui distingue le core d'une simple liste de contraintes : il isole la **cause**, pas le contexte.
> Note port : ce carnet jumeau n'emploie que l'API brute de `Microsoft.Z3` (`AssertAndTrack` + `UnsatCore`) ; le DSL Z3.Linq (`Explain()`), specifique a la serie voisine Z3-Linq2Z3, n'est pas porte ici — l'API brute fait le meme travail.


In [6]:
// Core minimal : {x==1, x==2} ecarte x>=0 (innocente, non impliquee)
var ctx14 = new Context();
var w = ctx14.MkIntConst("w");
var s4 = ctx14.MkSolver();
var r0 = ctx14.MkBoolConst("w_eq_1");    // x == 1
var r1 = ctx14.MkBoolConst("w_ge_0");    // x >= 0  (innocente)
var r2 = ctx14.MkBoolConst("w_eq_2");    // x == 2
s4.AssertAndTrack(ctx14.MkEq(w, ctx14.MkInt(1)), r0);
s4.AssertAndTrack(ctx14.MkGe(w, ctx14.MkInt(0)), r1);
s4.AssertAndTrack(ctx14.MkEq(w, ctx14.MkInt(2)), r2);
Console.WriteLine("Check(assumptions) : " + s4.Check());
var core4 = s4.UnsatCore.Select(c => c.ToString()).OrderBy(c => c).ToArray();
Console.WriteLine("UnsatCore = { " + string.Join("  ", core4) + " }   (x>=0 exclu du core)");


Check(assumptions) : UNSATISFIABLE


UnsatCore = { w_eq_1  w_eq_2 }   (x>=0 exclu du core)


## 6. Ce que ce notebook a demontre

Trois lecons sur le noyau d'insatisfiabilite :
1. **Du verdict a l'explication** - `unsat` dit *qu'il n'y a pas* de solution ; `UnsatCore` dit *pourquoi* (lesquelles contraintes conflictuent).

2. **Minimalite** - le core est le sous-ensemble **minimal** responsable : retirer un seul élément le rend satisfiable. Les contraintes irrelevant (compatibles, redondantes) sont ecartees.

3. **Actionnable** - sur une specification surcontrainte, le core pointe exactement la (ou les) contrainte(s) a relacher, transformant un echec de modelisation en diagnostic chirurgical.



Cette capacite d'explication distingue un solveur d'un simple oracle booléen : il ne dit pas seulement *non*, il dit *a cause de quoi*.


## 7. Exercices

Les trois exercices ci-dessous vous font manipuler le UNSAT core sur des variantes : conflit cache parmi 4 contraintes (1), planning avec relachement (2), core minimal parmi 10 (3). Chaque stub est self-contained : les fonctions (`Context.MkIntConst`, `Context.MkBoolConst`, `Context.MkSolver`, `AssertAndTrack`) sont definies dans les sections précédentes.


### Exercice 1 - Core sur un conflit cache (2 coupables parmi 4)
Construisez 4 contraintes sur `z` dont 2 sont compatibles et 2 en conflit (ex : `z <= 3`, `z >= 0`, `z == 7`, `z != 9`). Obtenez le core et verifiez qu'il ne contient **que** les 2 coupables.

**Étapes** : `z = ctx.MkIntConst("z")`, 4 `AssertAndTrack` etiquetes, `Check()`, afficher `UnsatCore` (doit exclure les 2 compatibles).


In [7]:
// Exercice 1 : core sur un conflit cache (2 coupables parmi 4)
// TODO etudiant :
// Etape 1 : z = ctxEx1.MkIntConst("z") ; 4 contraintes etiquetees (2 compatibles, 2 en conflit)
// Etape 2 : s.AssertAndTrack(formule, ctxEx1.MkBoolConst("etiq")) pour chacune
// Etape 3 : s.Check() + s.UnsatCore -> doit contenir seulement les 2 coupables
var ctxEx1 = new Context();
string[] resultEx1 = null; // TODO etudiant : les etiquettes du core trouve
Console.WriteLine("Exercice 1 a completer : isolez les 2 contraintes coupables parmi 4");


Exercice 1 a completer : isolez les 2 contraintes coupables parmi 4


### Exercice 2 - Planning : identifier puis relacher la coupable
Encodez un planning sur `j` : (a) `j >= 8`, (b) `j <= 18`, (c) `j != 3`, (d) `j == 3`. Obtenez le core, identifiez la coupable, puis **relachez-la** (re-checkez le sous-ensemble sans elle) -> le système devient SATISFIABLE.

**Étapes** : core = {j!=3, j=3} ; retirer l'une des deux du solver ; re-`Check()` -> `SATISFIABLE`.


In [8]:
// Exercice 2 : planning, identifier puis relacher la contrainte coupable
// TODO etudiant :
// Etape 1 : encoder (a)(b)(c)(d) sur j, obtenir le core
// Etape 2 : core = {j!=3, j=3} -> choisir laquelle relacher
// Etape 3 : re-check sur le sous-ensemble SANS la coupable -> SATISFIABLE
var ctxEx2 = new Context();
Status? resultEx2 = null; // TODO etudiant : le statut apres relachement (SATISFIABLE attendu)
Console.WriteLine("Exercice 2 a completer : identifiez et relachez la contrainte coupable");


Exercice 2 a completer : identifiez et relachez la contrainte coupable


### Exercice 3 - Core minimal parmi 10 contraintes
Construisez 10 contraintes sur `k` : 3 conflictuelles (`k == 1`, `k == 2`, `k == 3`) + 7 compatibles (`k >= 0`, `k <= 100`, etc.). Le core doit valoir exactement **3** (les 3 incompatibles), les 7 compatibles etant ecartees.

**Étapes** : 10 `AssertAndTrack` ; `Check()` ; compter `UnsatCore.Length` -> attendu 3.


In [9]:
// Exercice 3 : core minimal parmi 10 contraintes
// TODO etudiant :
// Etape 1 : k = ctxEx3.MkIntConst("k") ; 10 contraintes (3 k==n incompatibles + 7 compatibles)
// Etape 2 : s.AssertAndTrack pour les 10
// Etape 3 : s.Check() + s.UnsatCore.Length -> doit valoir 3
var ctxEx3 = new Context();
int? nbCoupables = null; // TODO etudiant : la taille du core mesuree
Console.WriteLine("Exercice 3 a completer : mesurer |core| parmi 10 contraintes (attendu : 3)");


Exercice 3 a completer : mesurer |core| parmi 10 contraintes (attendu : 3)


## Conclusion
Le UNSAT core transforme le solveur d'oracle binaire en **outil de diagnostic**. La ou `unsat` dit seulement ' impossible ', `UnsatCore` dit ' impossible **a cause de ces contraintes-la** ' - le sous-ensemble minimal responsable du conflit. Sur une specification surcontrainte, ce diagnostic est directement actionnable : il pointe la contrainte a relacher.

Cette serie Z3-API (notebooks 01 a 13, et leurs jumeaux C#) couvre maintenant les **trois postures** du solveur : **decider** (sat/unsat, 01-11), **optimiser** (Optimize, 06/08), et ici **expliquer** (UnsatCore, 13). La ou l'ingenieur classique debug a l'aveugle un système insatisfiable, le solveur SMT isole chirurgicalement la cause - la différence entre un oracle et un partenaire de raisonnement.

Le compagnon [Z3-13b - le MUS](Z3-13b-UnsatCores-MUS-CSharp.ipynb) approfondit ce langage : core invisible a l'inspection, minimalite non garantie par Z3, et l'algorithme deletion-based qui calcule le sous-ensemble irreductible.

***

> Carnet issu de l'EPIC #1206.
